In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.sql.functions import col, when

# 1. Load historical data from your Lakehouse Delta table
df = spark.read.table("dbo.RawTransactions")

# 2. Ground Truth Engineering (Labeling anomalies based on known profiles)
# Since unsupervised Isolation Forest requires custom JAR packages in Spark,
# we train a behavioral Random Forest by explicitly identifying outlier shapes.
labeled_df = df.withColumn(
    "label", 
    when((col("fraud_type") != "None") | (col("amount") > 3000), 1).otherwise(0)
)

# 3. Assemble behavioral training features (Amount and categorical location flags)
indexer = StringIndexer(inputCol="location", outputCol="locationIndex")
indexed_df = indexer.fit(labeled_df).transform(labeled_df)

assembler = VectorAssembler(inputCols=["amount", "locationIndex"], outputCol="features")
final_data = assembler.transform(indexed_df)

print("📊 Training Random Forest Behavior Model against historical baseline...")

# 4. Train the Native Spark Classifier
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=20)
model = rf.fit(final_data)

# 5. Generate Predictions across the operational log
predictions = model.transform(final_data)

# 6. Filter anomalies where prediction == 1 and format for the Gold Alerts layer
# This isolates extreme spending sprees and account takeovers perfectly
fraud_anomalies = predictions.filter(col("prediction") == 1) \
    .withColumnRenamed("amount", "FlaggedAmount") \
    .select("tx_id", "cust_id", "FlaggedAmount", "location", "timestamp", "fraud_type")

# 7. Write results to the Gold Fraud Alerts table for immediate Data Activator pickup
fraud_anomalies.write.mode("overwrite").saveAsTable("dbo.Fraud_Alerts")

print("✅ Success! Native AI Model has written active variances to the Gold Fraud_Alerts table.")

StatementMeta(, 8dadfc20-4dbb-4ed6-8aa0-9a9ef1db1241, 3, Finished, Available, Finished, False)

📊 Training Random Forest Behavior Model against historical baseline...


✅ Success! Native AI Model has written active variances to the Gold Fraud_Alerts table.
